# H1 EDA: 성장 정체와 접속 행동

현재 H1 입력 스키마, 결측, 성장 정체 비율, 주간 접속 관측 품질을 점검한다.
성장 정체 군집은 후보 확정 라벨이 아니다. 접속은 표본 통제변인(≥10/12)이며 클러스터링 피처가 아니다.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA = Path('../data')
features = pd.read_csv(DATA / 'features_monthly.csv', encoding='utf-8-sig')
raw = pd.read_csv(DATA / 'monthly_snapshots_raw.csv', encoding='utf-8-sig')

print('features:', features.shape, '| raw:', raw.shape)
print('raw 기간:', raw['year_month'].min(), '~', raw['year_month'].max())
assert {'access_active_weeks', 'access_observed_weeks'}.issubset(raw.columns)

features: (2000, 44) | raw: (24000, 12)
raw 기간: 2025-06 ~ 2026-05


## 1. H1 피처 품질

H1 채택 4-시스템 delta(전투력·헥사·유니온·어센틱 심볼)의 결측과 분포를 확인한다. (실제 클러스터링은 최근 6개월 재산출값을 쓰며, 여기선 12mo 집계 원천 분포를 점검한다.)

In [2]:
df = features[features['level'].between(270, 290)].copy()
# H1 채택 4피처의 원천 delta (cp_slog·hexa_avg·union_log·auth_log 의 변환 전 raw delta)
cols = ['avg_monthly_delta_combat_power', 'avg_monthly_delta_hexa',
        'avg_monthly_delta_union_level', 'avg_monthly_delta_authentic_symbol']
print('분석 표본:', len(df))
print('결측:')
print(df[cols].isna().sum().to_string())
print('\n분포:')
display(df[cols].describe().round(3))
print('\n음수 비율 (감소 = 정체/주차 신호):')
print((df[cols].lt(0).mean() * 100).round(1).astype(str).add('%').to_string())

분석 표본: 2000
결측:
avg_monthly_delta_combat_power        0
avg_monthly_delta_hexa                0
avg_monthly_delta_union_level         0
avg_monthly_delta_authentic_symbol    0

분포:


,avg_monthly_delta_combat_power,avg_monthly_delta_hexa,avg_monthly_delta_union_level,avg_monthly_delta_authentic_symbol
count,2.000000e+03,2000.000,2000.000,2000.000
mean,3.758810e+06,3.135,58.878,1.066
std,5.007917e+06,2.642,94.246,1.038
min,-2.545478e+07,-2.818,-17.000,-0.545
25%,1.179380e+06,1.364,20.614,0.364
50%,3.135375e+06,2.455,32.636,0.818
75%,5.640172e+06,4.091,47.447,1.455
max,5.191183e+07,22.091,781.091,6.000



음수 비율 (감소 = 정체/주차 신호):
avg_monthly_delta_combat_power        13.8%
avg_monthly_delta_hexa                 0.4%
avg_monthly_delta_union_level          0.1%
avg_monthly_delta_authentic_symbol     0.1%


## 2. 접속 관측 품질

월 1회 관측 대신 월내 4개 기준일을 조회한다. 원시 이력의 관측 주 수와 활동 주 수를 확인한다.

In [3]:
print('월별 access_flag:')
print(raw['access_flag'].value_counts(dropna=False).rename_axis('access_flag').to_string())
print('\n주간 관측 요약:')
display(raw[['access_active_weeks', 'access_observed_weeks']].describe().round(3))
print('\n캐릭터별 활동 요약:')
display(features[['access_active_months', 'access_active_weeks', 'access_observed_weeks', 'access_ratio']].describe().round(3))

# [접속 통제 검증] 재설계 표본은 >=10/12 접속 통제 -> access_active_months 는 통제변인(클러스터링 피처 아님).
m = features['access_active_months']
print('\n[접속 통제 검증] access_active_months >= 10 비율: '
      f'{(m >= 10).mean() * 100:.1f}%  (min={m.min():.0f}, max={m.max():.0f})')
print('  access_active_months 분포:', m.value_counts().sort_index().to_dict())

월별 access_flag:
access_flag
1.0    23178
0.0      701
NaN      121

주간 관측 요약:


,access_active_weeks,access_observed_weeks
count,23879.000,24000.000
mean,3.605,3.975
std,0.965,0.304
min,0.000,0.000
25%,4.000,4.000
50%,4.000,4.000
75%,4.000,4.000
max,4.000,4.000



캐릭터별 활동 요약:


,access_active_months,access_active_weeks,access_observed_weeks,access_ratio
count,2000.000,2000.000,2000.000,2000.000
mean,11.589,43.038,47.698,0.903
std,0.716,6.251,1.549,0.129
min,10.000,16.000,37.000,0.333
25%,11.000,39.000,48.000,0.833
50%,12.000,46.000,48.000,0.979
75%,12.000,48.000,48.000,1.000
max,12.000,48.000,48.000,1.000



[접속 통제 검증] access_active_months >= 10 비율: 100.0%  (min=10, max=12)
  access_active_months 분포: {10: 270, 11: 282, 12: 1448}


## 3. 현재 해석

- 성장 정체 군집과 주차 후보는 구분한다(확정 주차 유저 아님 — 공개 API에 보스/메소 기록 없음).
- H1은 최근 6개월(2025-12~2026-05) 4-시스템 delta(전투력·헥사·유니온·심볼)로 클러스터링해 `is_stagnant_cluster`(380명)를 만든다.
- H2/H3는 `data/cluster_labels.csv`의 `is_stagnant_cluster` 단일 라벨을 입력으로 쓴다. (시간분할 현재성 라벨은 폐기 — `temporal_external_validation` 삭제.)